In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
from natsort import natsorted
from tqdm.notebook import tqdm
from os.path import join as pjoin
import plotly.graph_objects as go
from scipy.stats import pearsonr, spearmanr, zscore, skew
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('/home/austinbaggetta/csstorage3/CircleTrack/CircleTrackAnalysis')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/place_fields'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#00802d', 'B': '#006c79', 'C': '#004da4', 'D': '#430073'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
male_mice = ['mc44', 'mc46', 'mc54', 'mc55', 'mc64', 'mc65']
control_mice = ['mc46', 'mc49', 'mc52', 'mc54', 'mc59', 'mc60', 'mc61', 'mc64']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.16 ## in radians
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'aligned_place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

### Example mouse. Create a distribution of place fields using the maximum of each tuning curve. Assumes each tuning curve has one peak that is central to that neuron's place field.

In [ ]:
## Set mouse information
experiment = 'MultiCon_Imaging6'
mouse = 'mc55'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'place_cells'
window_size = 6 ## number of spatial bins to use for plotting around reward location
rel_bins = np.arange(-(bin_size * window_size), (bin_size * window_size) + bin_size, bin_size)

In [ ]:
## Load and process data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass
neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Make Reward 1 the first rewarding port the mouse got water from
first_rew = sdata['lick_port'][sdata['water']].values[0]
if first_rew == sdata.attrs['reward_one']:
    first_rw_pos = reward_one_pos 
    second_rw_pos = reward_two_pos
else:
    first_rw_pos = reward_two_pos 
    second_rw_pos = reward_one_pos

h_shift_one, h_shift_two, H, xbin, mid_bin, rel_rw_one, rel_rw_two = pc.pf_relative_reward(tuning_curves, first_rw_pos, second_rw_pos, bins=bins[:-1], proportion=True)

In [ ]:
## Plot distribution of place fields across the track
fig = pf.custom_graph_template(x_title='Spatial Bin (rad)', y_title='Number of Place Fields')
fig.add_trace(go.Bar(x=xbin, y=H, marker_color='darkgrey', marker_line_width=2, marker_line_color='black'))
for val in [reward_one_pos, reward_two_pos]:
    fig.add_vline(x=val, line_dash='dash', line_width=1, opacity=1, line_color='red')
fig.show()
# fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_distribution_of_place_fields_running{only_running}_cordir{correct_dir}.png'), width=500, height=500)

In [ ]:
indices = np.arange(mid_bin - window_size, mid_bin + window_size + 1)
fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=2, 
                               titles=['Reward One', 'Reward Two'], width=800, shared_y=True, shared_x=True)
fig.add_trace(go.Bar(x=rel_bins, y=h_shift_one[indices], showlegend=False,
                     marker_color='darkgrey', marker_line_width=2, marker_line_color='black'), row=1, col=1)
fig.add_trace(go.Bar(x=rel_bins, y=h_shift_two[indices], showlegend=False,
                     marker_color='darkgrey', marker_line_width=2, marker_line_color='black'), row=1, col=2)
fig.update_yaxes(title='Proportion', col=1)
fig.show()
# fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_pf_relative_to_reward.png'), width=800, height=500)

### Combine across mice.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'
window_size = 6 ## number of spatial bins
rel_bins = np.arange(-(bin_size * window_size), (bin_size * window_size) + bin_size, bin_size)

In [ ]:
## Get proportion of place field peaks across spatial bins relative to reward.
## Makes reward 1 the first reward port the mouse received water from
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'proportion_one': [], 'proportion_two': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                index = ctn.mouse_indices(mouse, index)
                sdata = xr.open_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                else:
                    pass
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos

                h_shift_one, h_shift_two, H, xbin, mid_bin, rel_rw_one, rel_rw_two = pc.pf_relative_reward(tuning_curves, first_rw_pos, second_rw_pos, bins=bins[:-1], proportion=True)
                indices = np.arange(mid_bin - window_size, mid_bin + window_size + 1)

                for i, rbin in enumerate(rel_bins):
                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(index + 1)
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(rbin)
                    output_dict['proportion_one'].append(h_shift_one[indices[i]])
                    output_dict['proportion_two'].append(h_shift_two[indices[i]])
rel_df = pd.DataFrame(output_dict)
rel_df.to_feather(pjoin(int_data, f'max_pf_relative_rewards_{cell_type}.feat'))

In [ ]:
## Probability of a place field within each spatial bin for multi-context and two-context on day 16 separated by reward
cell_type = 'place_cells' 
day = 16
if cell_type == 'place_cells':
    rel_df = pd.read_feather(pjoin(int_data, 'max_pf_relative_rewards.feat'))
elif cell_type == 'all_cells':
    rel_df = pd.read_feather(pjoin(int_data, 'max_pf_relative_rewards_all_cells.feat'))
avg_df = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

d = avg_df[avg_df['day'] == day]
fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=2, width=1000,
                               shared_x=True, shared_y=True, titles=['Reward 1', 'Reward 2'])
for group in ['Two-context', 'Multi-context']:
    gdata = d[d['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['relative_bin'], y=gdata['proportion_one']['mean'], showlegend=False,
                         legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=gdata['proportion_one']['sem'])), row=1, col=1)
    fig.add_trace(go.Scattergl(x=gdata['relative_bin'], y=gdata['proportion_two']['mean'], showlegend=False,
                         legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=gdata['proportion_two']['sem'])), row=1, col=2)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.update_yaxes(range=[0, 0.08])
fig.data[0]['showlegend'] = True
fig.data[2]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, f'{day}_max_place_fields_relative_rw1_rw2_both_groups_{cell_type}.png'), width=1000, height=500)

In [ ]:
## Probability of a place field within each spatial bin for multi-context and two-context on day 16 both rewards combined
d = avg_df[avg_df['day'] == day]
fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='Proportion Place Fields', width=600)
for group in ['Two-context', 'Multi-context']:
    gdata = d[d['group'] == group]
    combined_data = {'relative_bin': [], 'prob': [], 'sem': []}
    for rbin in gdata['relative_bin'].unique():
        sub = gdata[gdata['relative_bin'] == rbin]
        combined_data['relative_bin'].append(rbin)
        combined_data['prob'].append(np.mean((sub['proportion_one']['mean'].values[0], sub['proportion_two']['mean'].values[0])))
        combined_data['sem'].append(np.mean((sub['proportion_one']['sem'].values[0], sub['proportion_two']['sem'].values[0])))
    fig.add_trace(go.Scattergl(x=combined_data['relative_bin'], y=combined_data['prob'], showlegend=True,
                         legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=combined_data['sem'])))
fig.update_yaxes(range=[0, 0.08])
fig.show()
fig.write_image(pjoin(fig_path, f'{day}_max_place_fields_relative_rewards_combined_both_groups_{cell_type}.png'), width=600, height=500)

In [ ]:
## Look at development of place field representation combining both groups in A
all_avg = rel_df.groupby(['day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=5, width=1400, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(1, 6)])
rw_colors = ['darkgrey', 'darkorchid']
contA = all_avg[all_avg['day'] < 6] ## subset for days just in context A
for day in contA['day'].unique():
    d_data = contA[contA['day'] == day]
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'], y=d_data['proportion_one']['mean'], showlegend=False,
                         legendgroup='Reward 1', name='Reward 1', marker_color=rw_colors[0], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=d_data['proportion_one']['sem'])), row=1, col=day)
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'], y=d_data['proportion_two']['mean'], showlegend=False,
                         legendgroup='Reward 2', name='Reward 2', marker_color=rw_colors[1], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=d_data['proportion_two']['sem'])), row=1, col=day)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.update_yaxes(range=[0, 0.08])
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=1.3,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Reward Distance (rad)':
        fig['layout']['annotations'][idx]['yshift'] = -40
fig.show()
fig.write_image(pjoin(fig_path, f'rw1_rw2_all_mice_contA_{cell_type}.png'), width=1400, height=500)

In [ ]:
## Look at development of place field representation in B for Multi-context and Two-context when combining reward locations
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=5, width=1400, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(6, 11)])
contB = avg[(avg['day'] >= 6) & (avg['day'] < 11)] ## subset for days just in context A
for day in contB['day'].unique():
    for group in ['Two-context', 'Multi-context']:
        d_data = contB[(contB['day'] == day) & (contB['group'] == group)]
        combined_data = {'relative_bin': [], 'prob': [], 'sem': []}
        for rbin in d_data['relative_bin'].unique():
            sub = d_data[d_data['relative_bin'] == rbin]
            combined_data['relative_bin'].append(rbin)
            combined_data['prob'].append(np.mean((sub['proportion_one']['mean'].values[0], sub['proportion_two']['mean'].values[0])))
            combined_data['sem'].append(np.mean((sub['proportion_one']['sem'].values[0], sub['proportion_two']['sem'].values[0])))
        fig.add_trace(go.Scattergl(x=combined_data['relative_bin'], y=combined_data['prob'], showlegend=False,
                            legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                            marker_line_color='black', error_y=dict(type='data', array=combined_data['sem'])), row=1, col=day - 5)
fig.update_yaxes(title='Proportion Place Fields', col=1)
fig.update_yaxes(range=[0, 0.08])
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=1.3,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Reward Distance (rad)':
        fig['layout']['annotations'][idx]['yshift'] = -40
fig.show()
fig.write_image(pjoin(fig_path, 'contB_rw_rep_both_groups.png'), width=1400, height=500)

In [ ]:
## Look at development of place field representation in B for Multi-context and Two-context when combining reward locations
avg = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=2, columns=5, width=1400, height=800, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(6, 11)])
contB = avg[(avg['day'] >= 6) & (avg['day'] < 11)] ## subset for days just in context A
for day in contB['day'].unique():
    for group in ['Two-context', 'Multi-context']:
        d_data = contB[(contB['day'] == day) & (contB['group'] == group)]
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'], y=d_data['proportion_one']['mean'], showlegend=False,
                                   legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                                   marker_line_color='black', error_y=dict(type='data', array=d_data['proportion_one']['sem'])), row=1, col=day - 5)
        fig.add_trace(go.Scattergl(x=d_data['relative_bin'], y=d_data['proportion_two']['mean'], showlegend=False,
                                   legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                                   marker_line_color='black', error_y=dict(type='data', array=d_data['proportion_two']['sem'])), row=2, col=day - 5)
fig.update_yaxes(title='Proportion Reward 1', row=1, col=1)
fig.update_yaxes(title='Proportion Reward 2', row=2, col=1)
fig.update_yaxes(range=[0, 0.1])
fig.data[0]['showlegend'] = True
fig.data[2]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=1.3,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Reward Distance (rad)':
        fig['layout']['annotations'][idx]['yshift'] = -40
fig.show()
fig.write_image(pjoin(fig_path, 'contB_rw1_rw2_rep_both_groups.png'), width=1400, height=800)

### Example mouse. Using the entire tuning curve for each neuron to create one tuning curve that represents coding across the track. Does not rely on any assumption about single vs multiple place fields.

In [ ]:
## Set mouse information
experiment = 'MultiCon_Imaging6'
mouse = 'mc55'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'place_cells'
tc_metric = 'above_zero' 
window_size = 6 ## number of spatial bins around reward
rel_bins = np.arange(-(bin_size * window_size), (bin_size * window_size) + bin_size, bin_size)

In [ ]:
## Load and process data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass
neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Make Reward 1 the first rewarding port the mouse got water from
first_rew = sdata['lick_port'][sdata['water']].values[0]
if first_rew == sdata.attrs['reward_one']:
    first_rw_pos = reward_one_pos 
    second_rw_pos = reward_two_pos
else:
    first_rw_pos = reward_two_pos 
    second_rw_pos = reward_one_pos

h_shift_one, h_shift_two, tc, x_bins, mid_bin, rel_rw_one, rel_rw_two = pc.tc_relative_reward(tuning_curves, first_rw_pos, second_rw_pos, bins=bins[:-1], tc_metric=tc_metric)

In [ ]:
## Plot activity relative to reward locations for a single mouse
indices = np.arange(mid_bin - window_size, mid_bin + window_size + 1)
fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=2, 
                               titles=['Reward One', 'Reward Two'], width=800, shared_y=True, shared_x=True)
fig.add_trace(go.Bar(x=rel_bins, y=h_shift_one[indices], showlegend=False,
                     marker_color='darkgrey', marker_line_width=2, marker_line_color='black'), row=1, col=1)
fig.add_trace(go.Bar(x=rel_bins, y=h_shift_two[indices], showlegend=False,
                     marker_color='darkgrey', marker_line_width=2, marker_line_color='black'), row=1, col=2)
if tc_metric == 'average':
    fig.update_yaxes(title='Average Spatial Activity', col=1)
elif tc_metric == 'above_zero':
    fig.update_yaxes(title='Proportion', col=1)
fig.show()

In [ ]:
## Plot average spatial activity across spatial bins
plot_all_ports = False
avg_tc = np.mean(tuning_curves, axis=0)
fig = pf.custom_graph_template(x_title='Spatial Bin (rad)', y_title='Average Spatial Activity')
fig.add_trace(go.Scattergl(x=bins[:-1], y=avg_tc, mode='lines+markers', line_color='darkgrey'))

if plot_all_ports:
    port_pos = []
    for value in np.unique(sdata['lick_port'].values):
        if value == -1:
            pass 
        else:
            port_pos.append(np.mean(sdata['lin_position'][(sdata['lick_port'] == value) & (sdata['correct_dir'])].values))

    for val in port_pos:
        if (val == reward_one_pos) | (val == reward_two_pos):
            c = 'red' 
        else:
            c = 'black'
        fig.add_vline(x=val, line_dash='dash', line_width=2, opacity=1, line_color=c)
else:
    for val in [reward_one_pos, reward_two_pos]:
        fig.add_vline(x=val, line_dash='dash', line_width=2, opacity=1, line_color='red')
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_avg_tuning_curve.png'), width=500, height=500)

In [ ]:
## Plot proportion of cells that fired in that spatial bin
plot_all_ports = False
total_tc = np.sum(tuning_curves > 1e-10, axis=0) ## have to use a small value because of the epsilon in spatial_activity function for decoding
total_tc = total_tc / np.sum(total_tc) ## normalize into proportion
fig = pf.custom_graph_template(x_title='Spatial Bin (rad)', y_title='Proportion')
fig.add_trace(go.Scattergl(x=bins[:-1], y=total_tc, mode='lines+markers', line_color='darkgrey'))

if plot_all_ports:
    port_pos = []
    for value in np.unique(sdata['lick_port'].values):
        if value == -1:
            pass 
        else:
            port_pos.append(np.mean(sdata['lin_position'][(sdata['lick_port'] == value) & (sdata['correct_dir'])].values))

    for val in port_pos:
        if (val == reward_one_pos) | (val == reward_two_pos):
            c = 'red' 
        else:
            c = 'black'
        fig.add_vline(x=val, line_dash='dash', line_width=2, opacity=1, line_color=c)
else:
    for val in [reward_one_pos, reward_two_pos]:
        fig.add_vline(x=val, line_dash='dash', line_width=2, opacity=1, line_color='red')
fig.update_yaxes(range=[0, 0.05])
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_proportion_tuning_curve.png'), width=500, height=500)

### Combine mice for average and above_zero metrics.

In [ ]:
## Settings
only_running = True
correct_dir = True
cell_type = 'all_cells'
tc_metric = 'above_zero' 
window_size = 6 ## number of spatial bins around reward
rel_bins = np.arange(-(bin_size * window_size), (bin_size * window_size) + bin_size, bin_size)

In [ ]:
## Get either average activity of each spatial bin or number of cells that were active in each spatial bin.
## Makes reward 1 the first reward port the mouse received water from
output_dict = {'mouse': [], 'group': [], 'sex': [], 'day': [], 'session': [], 'relative_bin': [], 'proportion_one': [], 'proportion_two': []}
for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                index = ctn.mouse_indices(mouse, index)
                sdata = xr.open_dataset(pjoin(mpath, session))[data_type]
                sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
                if cell_type == 'place_cells':
                    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                elif cell_type == 'nonplace_cells':
                    sdata = sdata[~sdata['skaggs_place'], :]
                else:
                    pass
                neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                velocity_thresh=velocity_thresh)
                ## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
                population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
                active_cells = np.sum(population_activity, axis=0) != 0
                population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
                tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                tuning_curves = tuning_curves.T ## cells x spatial bin

                ## Get reward positions
                reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
                reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

                ## Make Reward 1 the first rewarding port the mouse got water from
                first_rew = sdata['lick_port'][sdata['water']].values[0]
                if first_rew == sdata.attrs['reward_one']:
                    first_rw_pos = reward_one_pos 
                    second_rw_pos = reward_two_pos
                else:
                    first_rw_pos = reward_two_pos 
                    second_rw_pos = reward_one_pos

                h_shift_one, h_shift_two, H, xbin, mid_bin, rel_rw_one, rel_rw_two = pc.tc_relative_reward(tuning_curves, first_rw_pos, second_rw_pos, bins=bins[:-1], proportion=True, tc_metric=tc_metric)
                indices = np.arange(mid_bin - window_size, mid_bin + window_size + 1)

                for i, rbin in enumerate(rel_bins):
                    output_dict['mouse'].append(mouse)
                    output_dict['group'].append(sdata.attrs['group'])
                    output_dict['sex'].append(sdata.attrs['sex'])
                    output_dict['day'].append(index + 1)
                    output_dict['session'].append(sdata.attrs['session_two'])
                    output_dict['relative_bin'].append(rbin)
                    output_dict['proportion_one'].append(h_shift_one[indices[i]])
                    output_dict['proportion_two'].append(h_shift_two[indices[i]])
rel_df = pd.DataFrame(output_dict)
rel_df.to_feather(pjoin(int_data, f'{tc_metric}_relative_rewards_{cell_type}.feat'))

In [ ]:
## Proportion of cells with activity within each spatial bin for multi-context and two-context on day 16 separated by reward
day = 16
cell_type = 'place_cells'
if cell_type == 'place_cells':
    rel_df = pd.read_feather(pjoin(int_data, 'above_zero_relative_rewards.feat'))
elif cell_type == 'all_cells':
    rel_df = pd.read_feather(pjoin(int_data, 'above_zero_relative_rewards_all_cells.feat'))
avg_df = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

d = avg_df[avg_df['day'] == day]
fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=2, width=1000,
                               shared_x=True, shared_y=True, titles=['Reward One', 'Reward Two'])
for group in ['Two-context', 'Multi-context']:
    gdata = d[d['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['relative_bin'], y=gdata['proportion_one']['mean'], showlegend=False,
                         legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=gdata['proportion_one']['sem'])), row=1, col=1)
    fig.add_trace(go.Scattergl(x=gdata['relative_bin'], y=gdata['proportion_two']['mean'], showlegend=False,
                         legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=gdata['proportion_two']['sem'])), row=1, col=2)
fig.update_yaxes(title='Proportion', col=1)
fig.update_yaxes(range=[0, 0.04], dtick=0.01)
fig.data[0]['showlegend'] = True
fig.data[2]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, f'{day}_above_zero_relative_rw1_rw2_both_groups_{cell_type}.png'), width=1000, height=500)

In [ ]:
## Look at development of place field representation combining both groups in A
all_avg = rel_df.groupby(['day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=5, width=1400, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(1, 6)])
rw_colors = ['darkgrey', 'darkorchid']
contA = all_avg[all_avg['day'] < 6] ## subset for days just in context A
for day in contA['day'].unique():
    d_data = contA[contA['day'] == day]
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'], y=d_data['proportion_one']['mean'], showlegend=False,
                         legendgroup='Reward 1', name='Reward 1', marker_color=rw_colors[0], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=d_data['proportion_one']['sem'])), row=1, col=day)
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'], y=d_data['proportion_two']['mean'], showlegend=False,
                         legendgroup='Reward 2', name='Reward 2', marker_color=rw_colors[1], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=d_data['proportion_two']['sem'])), row=1, col=day)
fig.update_yaxes(title='Proportion', col=1)
fig.update_yaxes(range=[0, 0.05])
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=1.3,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Reward Distance (rad)':
        fig['layout']['annotations'][idx]['yshift'] = -40
fig.show()
fig.write_image(pjoin(fig_path, f'above_zero_rw1_rw2_all_mice_contA_{cell_type}.png'), width=1400, height=500)

In [ ]:
## Average activity within each spatial bin for multi-context and two-context on day 16 separated by reward
rel_df = pd.read_feather(pjoin(int_data, 'average_relative_rewards_all_cells.feat'))
avg_df = rel_df.groupby(['group', 'day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

d = avg_df[avg_df['day'] == 16]
fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=2, width=1000,
                               shared_x=True, shared_y=True, titles=['Reward One', 'Reward Two'])
for group in ['Two-context', 'Multi-context']:
    gdata = d[d['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['relative_bin'], y=gdata['proportion_one']['mean'], showlegend=False,
                         legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=gdata['proportion_one']['sem'])), row=1, col=1)
    fig.add_trace(go.Scattergl(x=gdata['relative_bin'], y=gdata['proportion_two']['mean'], showlegend=False,
                         legendgroup=group, name=group, marker_color=ce_colors_dict[group], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=gdata['proportion_two']['sem'])), row=1, col=2)
fig.update_yaxes(title='Activity', col=1)
fig.data[0]['showlegend'] = True
fig.data[2]['showlegend'] = True
fig.show()
fig.write_image(pjoin(fig_path, f'average_act_relative_rw1_rw2_both_groups_{cell_type}.png'), width=1000, height=500)

In [ ]:
## Look at development of activity around rewards combining both groups in A
all_avg = rel_df.groupby(['day', 'relative_bin'], as_index=False).agg({'proportion_one': ['mean', 'sem'], 'proportion_two': ['mean', 'sem']})

fig = pf.custom_graph_template(x_title='Reward Distance (rad)', y_title='', rows=1, columns=5, width=1400, master_axes=True,
                               shared_x=True, shared_y=True, titles=[f'Day {x}' for x in np.arange(1, 6)])
rw_colors = ['darkgrey', 'darkorchid']
contA = all_avg[all_avg['day'] < 6] ## subset for days just in context A
for day in contA['day'].unique():
    d_data = contA[contA['day'] == day]
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'], y=d_data['proportion_one']['mean'], showlegend=False,
                         legendgroup='Reward 1', name='Reward 1', marker_color=rw_colors[0], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=d_data['proportion_one']['sem'])), row=1, col=day)
    fig.add_trace(go.Scattergl(x=d_data['relative_bin'], y=d_data['proportion_two']['mean'], showlegend=False,
                         legendgroup='Reward 2', name='Reward 2', marker_color=rw_colors[1], marker_line_width=2,
                         marker_line_color='black', error_y=dict(type='data', array=d_data['proportion_two']['sem'])), row=1, col=day)
fig.update_yaxes(title='Activity', col=1)
fig.data[0]['showlegend'] = True
fig.data[1]['showlegend'] = True
fig.update_layout(legend=dict(
    orientation="h",
    yanchor="top",
    y=1.3,
    xanchor="center",
    x=0.48
))
for idx, annotation in enumerate(fig['layout']['annotations']):
    if annotation['text'] == 'Reward Distance (rad)':
        fig['layout']['annotations'][idx]['yshift'] = -40
fig.show()
fig.write_image(pjoin(fig_path, f'average_rw1_rw2_all_mice_contA_{cell_type}.png'), width=1400, height=500)